# Nasdaq ETF Testing

Download the current Nasdaq-listed exchange-traded products table and explore it in a sortable, filterable Plotly Dash table.

In [ ]:
%pip install -q pandas plotly dash lxml openpyxl requests

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from dash import Dash, dash_table, dcc, html

def find_project_root():
    """Find the nearest parent containing the Quantapp package."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / "Quantapp").is_dir():
            return directory
    raise FileNotFoundError("Could not find the Investment Research project root.")

PROJECT_ROOT = find_project_root()
DATA_DIRECTORY = PROJECT_ROOT / "data" / "market_listings"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
SOURCE_URL = "https://www.nasdaqtrader.com/trader.aspx?id=etf_definitions"
OUTPUT_FILE = DATA_DIRECTORY / "nasdaq_listed_etps.csv"

## Download and clean the Nasdaq table

Nasdaq publishes one row per ETF/liquidity-provider relationship, so a ticker can appear more than once. The full source data is retained.

In [ ]:
tables = pd.read_html(SOURCE_URL)

# Locate the ETF table by its Symbol column instead of relying on a fragile table index.
def flatten_columns(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [" ".join(str(part) for part in col if not str(part).startswith("Unnamed")).strip() for col in frame.columns]
    else:
        frame.columns = [str(col).strip() for col in frame.columns]
    return frame

tables = [flatten_columns(table) for table in tables]
candidates = [table for table in tables if "Symbol" in table.columns and any("Fund" in col for col in table.columns)]
if not candidates:
    raise RuntimeError("Could not locate Nasdaq's ETF table; the page structure may have changed.")

etfs = max(candidates, key=len).dropna(how="all").reset_index(drop=True)
etfs.columns = [col.replace("\xa0", " ").strip() for col in etfs.columns]
for column in etfs.select_dtypes(include="object"):
    etfs[column] = etfs[column].str.replace("\xa0", " ", regex=False).str.strip()

etfs.to_csv(OUTPUT_FILE, index=False)
print(f"Downloaded {len(etfs):,} Nasdaq ETF records across {etfs['Symbol'].nunique():,} unique symbols.")
print(f"Saved a local copy to: {OUTPUT_FILE.resolve()}")

## Interactive table

Click a column heading to sort. Use the boxes below the headings to filter; combine text filters with operators such as `contains`, `=`, `>`, and `<`.

In [ ]:
app = Dash(__name__)
app.layout = html.Div(
    [
        html.H3("Nasdaq-Listed Exchange-Traded Products"),
        html.P(f"{len(etfs):,} records | {etfs['Symbol'].nunique():,} unique symbols"),
        dash_table.DataTable(
            data=etfs.where(pd.notna(etfs), None).to_dict("records"),
            columns=[{"name": column, "id": column} for column in etfs.columns],
            sort_action="native",
            sort_mode="multi",
            filter_action="native",
            page_action="native",
            page_size=25,
            fixed_rows={"headers": True},
            style_table={"height": "650px", "overflowX": "auto", "overflowY": "auto"},
            style_cell={
                "fontFamily": "Arial, sans-serif",
                "fontSize": 13,
                "padding": "8px",
                "textAlign": "left",
                "minWidth": "110px",
                "maxWidth": "300px",
                "whiteSpace": "normal",
            },
            style_header={"backgroundColor": "#0b1f3a", "color": "white", "fontWeight": "bold"},
            style_filter={"backgroundColor": "#eef3f8"},
            style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f8fafc"}],
        ),
    ],
    style={"padding": "12px"},
)

app.run(jupyter_mode="inline", port=8050, debug=False)

## NYSE Arca ETF Lead Market Makers

Download NYSE Arca's current ETP/LMM workbook, retain every published record, and save a reusable local CSV copy.

In [ ]:
NYSE_SOURCE_URL = "https://www.nyse.com/publicdocs/nyse/markets/nyse-arca/NYSE_Arca_Equities_LMM_Current.xlsx"
NYSE_OUTPUT_FILE = DATA_DIRECTORY / "nyse_arca_etp_lmm_current.csv"

nyse_etfs = pd.read_excel(NYSE_SOURCE_URL, engine="openpyxl").dropna(how="all").reset_index(drop=True)
nyse_etfs.columns = [str(column).replace("\xa0", " ").strip() for column in nyse_etfs.columns]
for column in nyse_etfs.select_dtypes(include="object"):
    nyse_etfs[column] = nyse_etfs[column].str.replace("\xa0", " ", regex=False).str.strip()

required_columns = {"Symbol", "ETP Name", "Product Category", "Lead Market Maker"}
missing_columns = required_columns.difference(nyse_etfs.columns)
if missing_columns:
    raise RuntimeError(f"NYSE workbook format changed; missing columns: {sorted(missing_columns)}")

nyse_etfs.to_csv(NYSE_OUTPUT_FILE, index=False)
print(f"Downloaded {len(nyse_etfs):,} NYSE Arca ETP records across {nyse_etfs['Symbol'].nunique():,} unique symbols.")
print(f"Saved a local copy to: {NYSE_OUTPUT_FILE.resolve()}")

### Interactive NYSE Arca table

Click a heading to sort, Shift-click for multi-column sorting, or use the filter boxes below the headings.

In [ ]:
nyse_app = Dash("nyse_arca_etf_table")
nyse_app.layout = html.Div(
    [
        html.H3("NYSE Arca ETP Lead Market Makers"),
        html.P(f"{len(nyse_etfs):,} records | {nyse_etfs['Symbol'].nunique():,} unique symbols"),
        dash_table.DataTable(
            data=nyse_etfs.where(pd.notna(nyse_etfs), None).to_dict("records"),
            columns=[{"name": column, "id": column} for column in nyse_etfs.columns],
            sort_action="native",
            sort_mode="multi",
            filter_action="native",
            page_action="native",
            page_size=25,
            fixed_rows={"headers": True},
            style_table={"height": "650px", "overflowX": "auto", "overflowY": "auto"},
            style_cell={
                "fontFamily": "Arial, sans-serif",
                "fontSize": 13,
                "padding": "8px",
                "textAlign": "left",
                "minWidth": "120px",
                "maxWidth": "420px",
                "whiteSpace": "normal",
            },
            style_header={"backgroundColor": "#003b5c", "color": "white", "fontWeight": "bold"},
            style_filter={"backgroundColor": "#e8f1f5"},
            style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f7fafb"}],
        ),
    ],
    style={"padding": "12px"},
)

nyse_app.run(jupyter_mode="inline", port=8051, debug=False)

## Complete NYSE ETF Listings Directory

The public directory is paginated and populated by NYSE's JSON endpoint. This section requests every API page, verifies the downloaded row count against NYSE's reported total, and saves the latest complete snapshot in the project's `data/market_listings` directory.

In [ ]:
NYSE_DIRECTORY_PAGE_URL = "https://www.nyse.com/listings_directory/etf"
NYSE_DIRECTORY_API_URL = "https://www.nyse.com/api/quotes/filter"
NYSE_DIRECTORY_OUTPUT_FILE = DATA_DIRECTORY / "nyse_etf_listings_directory.csv"
ROWS_PER_REQUEST = 1_000

directory_records = []
reported_total = None
page_number = 1

with requests.Session() as session:
    while reported_total is None or len(directory_records) < reported_total:
        payload = {
            "instrumentType": "EXCHANGE_TRADED_FUND",
            "pageNumber": page_number,
            "sortColumn": "NORMALIZED_TICKER",
            "sortOrder": "ASC",
            "maxResultsPerPage": ROWS_PER_REQUEST,
            "filterToken": "",
        }
        response = session.post(NYSE_DIRECTORY_API_URL, json=payload, timeout=60)
        response.raise_for_status()
        page_records = response.json()
        if not page_records:
            break

        if reported_total is None:
            reported_total = int(page_records[0]["total"])
        directory_records.extend(page_records)
        print(f"Page {page_number}: {len(page_records):,} rows ({len(directory_records):,}/{reported_total:,})")
        page_number += 1

if reported_total is None or len(directory_records) != reported_total:
    raise RuntimeError(f"Incomplete NYSE download: received {len(directory_records):,} of {reported_total or 0:,} rows.")

nyse_directory = pd.DataFrame(directory_records).rename(
    columns={
        "normalizedTicker": "Ticker",
        "instrumentName": "Instrument Name",
        "symbolExchangeTicker": "Exchange Ticker",
        "url": "Quote URL",
    }
)
nyse_directory.insert(
    0,
    "Listing Exchange",
    nyse_directory["Quote URL"].str.extract(r"/quote/([^:]+):", expand=False),
)
display_columns = ["Listing Exchange", "Ticker", "Instrument Name", "Exchange Ticker", "Quote URL"]
nyse_directory = nyse_directory[display_columns]
nyse_directory.to_csv(NYSE_DIRECTORY_OUTPUT_FILE, index=False)

print(f"Downloaded all {len(nyse_directory):,} records from {page_number - 1} API pages.")
print(f"Saved the latest copy to: {NYSE_DIRECTORY_OUTPUT_FILE.resolve()}")

### Interactive complete NYSE directory

Click a heading to sort, Shift-click for multi-column sorting, or filter using the boxes beneath the headings.

In [ ]:
directory_app = Dash("nyse_complete_etf_directory")
directory_app.layout = html.Div(
    [
        html.H3("Complete NYSE ETF Listings Directory"),
        html.P(f"{len(nyse_directory):,} listing records"),
        dash_table.DataTable(
            data=nyse_directory.where(pd.notna(nyse_directory), None).to_dict("records"),
            columns=[{"name": column, "id": column} for column in nyse_directory.columns],
            sort_action="native",
            sort_mode="multi",
            filter_action="native",
            page_action="native",
            page_size=25,
            fixed_rows={"headers": True},
            style_table={"height": "650px", "overflowX": "auto", "overflowY": "auto"},
            style_cell={
                "fontFamily": "Arial, sans-serif",
                "fontSize": 13,
                "padding": "8px",
                "textAlign": "left",
                "minWidth": "120px",
                "maxWidth": "480px",
                "whiteSpace": "normal",
            },
            style_header={"backgroundColor": "#003b5c", "color": "white", "fontWeight": "bold"},
            style_filter={"backgroundColor": "#e8f1f5"},
            style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f7fafb"}],
        ),
    ],
    style={"padding": "12px"},
)

directory_app.run(jupyter_mode="inline", port=8052, debug=False)

## Nasdaq Symbol Directory Files

Download Nasdaq Traded, Other Listed, and Nasdaq Listed symbol directories. These feeds contain all securities and identify exchange-traded funds through the ETF column. The non-data File Creation Time trailer is captured separately and excluded from each CSV.

In [ ]:
SYMBOL_DIRECTORY_FEEDS = {
    "Nasdaq Traded": {
        "url": "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt",
        "output": DATA_DIRECTORY / "nasdaq_traded_symbols.csv",
    },
    "Other Listed": {
        "url": "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt",
        "output": DATA_DIRECTORY / "other_listed_symbols.csv",
    },
    "Nasdaq Listed": {
        "url": "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt",
        "output": DATA_DIRECTORY / "nasdaq_listed_symbols.csv",
    },
}

def download_symbol_directory(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    frame = pd.read_csv(StringIO(response.text), sep="|", dtype=str)
    frame.columns = [str(column).replace("\xa0", " ").strip() for column in frame.columns]

    first_column = frame.columns[0]
    trailer_mask = frame[first_column].fillna("").str.startswith("File Creation Time:")
    file_creation_time = frame.loc[trailer_mask, first_column].iloc[0] if trailer_mask.any() else "Not provided"
    frame = frame.loc[~trailer_mask].dropna(how="all").reset_index(drop=True)

    for column in frame.columns:
        frame[column] = frame[column].str.replace("\xa0", " ", regex=False).str.strip()
    return frame, file_creation_time

symbol_directories = {}
symbol_directory_metadata = {}

for feed_name, feed in SYMBOL_DIRECTORY_FEEDS.items():
    frame, creation_time = download_symbol_directory(feed["url"])
    frame.to_csv(feed["output"], index=False)
    symbol_directories[feed_name] = frame
    symbol_directory_metadata[feed_name] = creation_time
    etf_count = frame["ETF"].eq("Y").sum() if "ETF" in frame.columns else 0
    print(f"{feed_name}: {len(frame):,} securities | {etf_count:,} ETFs | {creation_time}")
    print(f"Saved to: {feed['output'].resolve()}")

### Interactive Nasdaq symbol directories

Select a feed tab, click a heading to sort, Shift-click for multi-column sorting, or filter beneath any heading. Filter the ETF column to Y to show only ETFs.

In [ ]:
def symbol_directory_table(frame):
    return dash_table.DataTable(
        data=frame.where(pd.notna(frame), None).to_dict("records"),
        columns=[{"name": column, "id": column} for column in frame.columns],
        sort_action="native",
        sort_mode="multi",
        filter_action="native",
        page_action="native",
        page_size=25,
        fixed_rows={"headers": True},
        style_table={"height": "650px", "overflowX": "auto", "overflowY": "auto"},
        style_cell={
            "fontFamily": "Arial, sans-serif",
            "fontSize": 13,
            "padding": "8px",
            "textAlign": "left",
            "minWidth": "110px",
            "maxWidth": "480px",
            "whiteSpace": "normal",
        },
        style_header={"backgroundColor": "#0b1f3a", "color": "white", "fontWeight": "bold"},
        style_filter={"backgroundColor": "#eef3f8"},
        style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f8fafc"}],
    )

symbol_app = Dash("nasdaq_symbol_directories")
symbol_app.layout = html.Div(
    [
        html.H3("Nasdaq Symbol Directories"),
        dcc.Tabs(
            [
                dcc.Tab(
                    label=f"{feed_name} ({len(frame):,})",
                    children=[
                        html.P(symbol_directory_metadata[feed_name], style={"margin": "12px 0"}),
                        symbol_directory_table(frame),
                    ],
                )
                for feed_name, frame in symbol_directories.items()
            ]
        ),
    ],
    style={"padding": "12px"},
)

symbol_app.run(jupyter_mode="inline", port=8053, debug=False)

## Super Mega Combined Table

Combine every downloaded dataset without discarding duplicate listings or liquidity-provider relationships. Unified Symbol and Unified Name provide consistent fields across sources, while Source Dataset and Source Link identify the exact origin of every row.

In [ ]:
MEGA_TABLE_OUTPUT_FILE = DATA_DIRECTORY / "combined_exchange_listings_mega_table.csv"

def prepare_for_mega_table(frame, dataset_name, source_link, symbol_column, name_column):
    prepared = frame.copy()
    prepared.insert(0, "Unified Name", prepared[name_column] if name_column in prepared.columns else pd.NA)
    prepared.insert(0, "Unified Symbol", prepared[symbol_column] if symbol_column in prepared.columns else pd.NA)
    prepared.insert(0, "Source Link", source_link)
    prepared.insert(0, "Source Dataset", dataset_name)
    return prepared

mega_sources = [
    prepare_for_mega_table(etfs, "Nasdaq Listed ETP Definitions", SOURCE_URL, "Symbol", "Fund"),
    prepare_for_mega_table(nyse_etfs, "NYSE Arca ETP Lead Market Makers", NYSE_SOURCE_URL, "Symbol", "ETP Name"),
    prepare_for_mega_table(nyse_directory, "NYSE ETF Listings Directory", NYSE_DIRECTORY_PAGE_URL, "Ticker", "Instrument Name"),
    prepare_for_mega_table(
        symbol_directories["Nasdaq Traded"],
        "Nasdaq Traded Symbol Directory",
        SYMBOL_DIRECTORY_FEEDS["Nasdaq Traded"]["url"],
        "Symbol",
        "Security Name",
    ),
    prepare_for_mega_table(
        symbol_directories["Other Listed"],
        "Other Listed Symbol Directory",
        SYMBOL_DIRECTORY_FEEDS["Other Listed"]["url"],
        "ACT Symbol",
        "Security Name",
    ),
    prepare_for_mega_table(
        symbol_directories["Nasdaq Listed"],
        "Nasdaq Listed Symbol Directory",
        SYMBOL_DIRECTORY_FEEDS["Nasdaq Listed"]["url"],
        "Symbol",
        "Security Name",
    ),
]

mega_table = pd.concat(mega_sources, ignore_index=True, sort=False)
priority_columns = ["Source Dataset", "Source Link", "Unified Symbol", "Unified Name"]
mega_table = mega_table[priority_columns + [column for column in mega_table.columns if column not in priority_columns]]
mega_table.to_csv(MEGA_TABLE_OUTPUT_FILE, index=False)

print(f"Combined {len(mega_table):,} source records across {mega_table['Unified Symbol'].nunique():,} unique normalized symbols.")
print(mega_table.groupby("Source Dataset").size().rename("Rows").to_string())
print(f"Saved to: {MEGA_TABLE_OUTPUT_FILE.resolve()}")

### Interactive super mega table

Sort or filter any field. Use Source Dataset to isolate a feed and Source Link to verify each record's origin.

In [ ]:
mega_app = Dash("combined_exchange_listings_mega_table")
mega_app.layout = html.Div(
    [
        html.H3("Super Mega Combined Exchange Listings Table"),
        html.P(f"{len(mega_table):,} source records | {mega_table['Unified Symbol'].nunique():,} unique symbols"),
        dash_table.DataTable(
            data=mega_table.where(pd.notna(mega_table), None).to_dict("records"),
            columns=[{"name": column, "id": column} for column in mega_table.columns],
            sort_action="native",
            sort_mode="multi",
            filter_action="native",
            page_action="native",
            page_size=25,
            fixed_rows={"headers": True},
            style_table={"height": "700px", "overflowX": "auto", "overflowY": "auto"},
            style_cell={
                "fontFamily": "Arial, sans-serif",
                "fontSize": 12,
                "padding": "7px",
                "textAlign": "left",
                "minWidth": "115px",
                "maxWidth": "480px",
                "whiteSpace": "normal",
            },
            style_header={"backgroundColor": "#071b33", "color": "white", "fontWeight": "bold"},
            style_filter={"backgroundColor": "#e7eef6"},
            style_data_conditional=[{"if": {"row_index": "odd"}, "backgroundColor": "#f7f9fc"}],
        ),
    ],
    style={"padding": "12px"},
)

mega_app.run(jupyter_mode="inline", port=8054, debug=False)